In [16]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge


In [17]:
# Load daily commodity prices
df = pd.read_csv("C:/Users/Rudradwivedi/Desktop/Matrisk AI/data/commodity_prices/DS2_commodity_prices_10yr.csv")

# Inspect
df.head()
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22952 entries, 0 to 22951
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                22952 non-null  object 
 1   commodity           22952 non-null  object 
 2   open                22952 non-null  float64
 3   high                22952 non-null  float64
 4   low                 22952 non-null  float64
 5   close               22952 non-null  float64
 6   volume              22952 non-null  int64  
 7   daily_return        22944 non-null  float64
 8   return_5d           22912 non-null  float64
 9   return_21d          22784 non-null  float64
 10  volatility_5d_ann   22912 non-null  float64
 11  volatility_21d_ann  22784 non-null  float64
 12  volatility_63d_ann  22448 non-null  float64
 13  sma_21              22792 non-null  float64
 14  sma_63              22456 non-null  float64
 15  bollinger_upper     22792 non-null  float64
 16  boll

In [18]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

# Focus on one commodity, e.g. Copper
df_copper = df[df['commodity'] == 'Copper'].copy()

In [19]:
features = [
    'open','high','low','volume','daily_return','return_5d','return_21d',
    'volatility_5d_ann','volatility_21d_ann','volatility_63d_ann',
    'sma_21','sma_63','bollinger_upper','bollinger_lower','bollinger_z',
    'rsi_14','macd','macd_signal','momentum_10d','momentum_21d','term_spread'
]

X = df_copper[features].dropna()
y = df_copper.loc[X.index, 'close']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)


In [22]:
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Test MSE:", mean_squared_error(y_test, y_pred))


Test MSE: 3.2979724460905746e-05


In [24]:
latest_features = X.iloc[-1].values.reshape(1,-1)
forecast = model.predict(latest_features)
print("Next day forecast:", forecast[0])


Next day forecast: 3.9067774229901433


C:\Users\Rudradwivedi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


In [26]:
df_copper.loc[X.index, 'predicted_close'] = model.predict(X)
df_copper.to_csv("C:/Users/Rudradwivedi/Desktop/Matrisk AI/data/commodity_prices/copper_forecast.csv", index=False)
